# Standard DNN - Gallstone Dataset\n\nTrains a standard (non-private) Deep Neural Network on the Gallstone dataset.


## 1. Import Libraries


In [1]:
import time
import numpy as np
import pandas as pd
import pickle
import json
import os
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Shared metric collector (repository root): logs training accuracy,
# balanced accuracy and AUROC alongside the metrics already tracked below.
import os as _os
import sys as _sys
_REPO_ROOT = _os.path.abspath(_os.path.join('..', '..'))
if _REPO_ROOT not in _sys.path:
    _sys.path.insert(0, _REPO_ROOT)
from metrics_utils import ExtraMetrics


Using device: cpu


## 2. Load Preprocessed Data


In [2]:
with open('../data/processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']

X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.LongTensor(y_test)

batch_size = 32
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


## 3. Define DNN Architecture


In [3]:
class DNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_sizes=[64, 32], num_classes=2, dropout=0.3):
        super(DNNClassifier, self).__init__()
        layers = []
        prev_size = input_size
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size
        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

input_size = X_train.shape[1]
hidden_sizes = [64, 32]
num_classes = 2
dropout = 0.3
learning_rate = 0.001
num_epochs = 50


## 4. Train standard model (Multiple Runs)


In [4]:
N_RUNS = 30
detail_accuracies = []
detail_f1s = []
detail_precisions = []
detail_recalls = []

def train_model(model, loader, criterion, optimizer, epochs, dev):
    model.train()
    for epoch in range(epochs):
        for inputs, labels in loader:
            inputs, labels = inputs.to(dev), labels.to(dev)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

def evaluate_model(model, loader, device):
    """Return (labels, predictions, class probabilities) for `loader`.

    The probabilities are what AUROC needs; predictions and labels are
    unchanged from the original implementation.
    """
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            all_probs.extend(torch.softmax(outputs, dim=1).cpu().numpy())
            all_preds.extend(torch.max(outputs, 1)[1].cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

print(f"Training Standard DNN {N_RUNS} times...")
print('='*80)

extra = ExtraMetrics()
for run in range(N_RUNS):
    seed = run * 10 + 42
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    model = DNNClassifier(input_size, hidden_sizes, num_classes, dropout).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    _t0 = time.perf_counter()
    train_model(model, train_loader, criterion, optimizer, num_epochs, device)
    train_time = time.perf_counter() - _t0
    # Training-partition score for the non-private fit.
    y_train_true, y_train_pred, _ = evaluate_model(model, train_loader, device)
    y_true, y_pred, y_prob = evaluate_model(model, test_loader, device)
    
    accuracy = accuracy_score(y_true, y_pred)
    extra.add(y_true, y_pred, y_score=y_prob, train_true=y_train_true, train_pred=y_train_pred, train_time=train_time)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    
    detail_accuracies.append(accuracy)
    detail_f1s.append(f1)
    detail_precisions.append(precision)
    detail_recalls.append(recall)
    
    print(f"  Run {run+1:2d}/{N_RUNS} | Acc: {accuracy:.4f} | F1: {f1:.4f}")

print('='*80)


Training Standard DNN 30 times...


  Run  1/30 | Acc: 0.7500 | F1: 0.7500
  Run  2/30 | Acc: 0.7500 | F1: 0.7576


  Run  3/30 | Acc: 0.7812 | F1: 0.7742
  Run  4/30 | Acc: 0.7969 | F1: 0.8000


  Run  5/30 | Acc: 0.8125 | F1: 0.8125


  Run  6/30 | Acc: 0.7969 | F1: 0.8000
  Run  7/30 | Acc: 0.7812 | F1: 0.7812


...

  Run 21/30 | Acc: 0.7969 | F1: 0.8000
  Run 22/30 | Acc: 0.7656 | F1: 0.7619


  Run 23/30 | Acc: 0.7656 | F1: 0.7692
  Run 24/30 | Acc: 0.7812 | F1: 0.7941


  Run 25/30 | Acc: 0.8125 | F1: 0.8235
  Run 26/30 | Acc: 0.7812 | F1: 0.7879


  Run 27/30 | Acc: 0.7500 | F1: 0.7419
  Run 28/30 | Acc: 0.7969 | F1: 0.8000


  Run 29/30 | Acc: 0.8125 | F1: 0.8125
  Run 30/30 | Acc: 0.7812 | F1: 0.7879


## 5. Aggregate & Save Baseline Results


In [5]:
accuracy = float(np.mean(detail_accuracies))
f1 = float(np.mean(detail_f1s))
precision = float(np.mean(detail_precisions))
recall = float(np.mean(detail_recalls))
std_accuracy = float(np.std(detail_accuracies))
std_f1 = float(np.std(detail_f1s))

os.makedirs('output', exist_ok=True)

baseline_results = {
    'model': 'Standard DNN',
    'dataset': 'Gallstone',
    'accuracy': accuracy,
    'f1_score': f1,
    'precision': precision,
    'recall': recall,
    'accuracy_std': std_accuracy,
    'f1_std': std_f1
}
baseline_results.update(extra.means())

pd.DataFrame([baseline_results]).to_csv('output/baseline_dnn_results.csv', index=False)
with open('output/std_dnn_report.json', 'w') as f:
    json.dump(baseline_results, f, indent=4)

print("Baseline results saved successfully.")

# Export model for MIA
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..', '..')))
from model_exporter import export_model
export_model(model, 'output/model/std_dnn_model.pt')


print("Added metrics:")
print(extra.describe())

Baseline results saved successfully.
PyTorch model successfully exported to output/model/std_dnn_model.pt
Added metrics:
  Train Acc:      0.8786 ± 0.0100
  Balanced Acc:   0.7828 ± 0.0226
  AUROC:          0.8469 ± 0.0123
  Train Time (s): 0.1425 ± 0.0334
